#  Zero-Shot vs. Fine-Tuned Summarization Comparison

This notebook demonstrates how to compare the performance of zero-shot prompting with instruction-tuned models vs. fine-tuned summarization models (like BART by Facebook).

In [ ]:
!pip install -q transformers datasets ipywidgets

In [ ]:
#  Sample: Simulated movie reviews
movie_reviews = """
The film was a visual masterpiece with stunning cinematography.
However, the plot felt slow and predictable.
Great performances by the lead actors.
The score complemented the tone perfectly.
It dragged a bit in the middle but the ending was satisfying.
"""

##  1. Zero-Shot Prompting Summarization (FLAN-T5)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time

zero_shot_prompt = f"""
Below are several movie reviews:
{movie_reviews}

Summarize the overall sentiment and main points in 3 sentences.
"""

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(
    zero_shot_prompt,
    return_tensors="pt",
    max_length=512,
    padding="max_length",
    truncation=True
)

start_time = time.time()
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)
end_time = time.time()

zero_shot_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Zero-shot Prompted Summary:\n")
print(zero_shot_summary)
print(f"FLAN-T5 Inference Time: {end_time - start_time:.2f} seconds")

## 2. Fine-Tuned Summarization (BART)

In [ ]:
from transformers import pipeline
import time

try:
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    start_time = time.time()
    summary = summarizer(movie_reviews, max_length=100, min_length=30, do_sample=False)
    end_time = time.time()
    print(" Fine-tuned Summarizer Output:\n")
    print(summary[0]['summary_text'])
    print(f"BART Inference Time: {end_time - start_time:.2f} seconds")
except Exception as e:
    print(" Error during summarization:", e)

###  Comparison: Zero-Shot vs. Fine-Tuned Summarizer

| Aspect | Zero-Shot Prompting (FLAN-T5) | Fine-Tuned Summarizer (BART-CNN) |
|--------|-------------------------------|-----------------------------------|
| Setup  | Instruction-based prompt | Task-specific pre-trained weights |
| Flexibility | Works on any task with clever prompts | Performs best on summarization |
| Output Style | More generic or varied | Concise and on-topic |
| Customization | Requires good prompt design | Requires labeled data for training |
| Inference Time | Typically slower | Optimized for summarization |
